In [ ]:

# --- 1. CONFIGURACIÓN DE LA PÁGINA ---
st.set_page_config(
    page_title="Predictor de Fútbol Pro", 
    page_icon="⚽",
    layout="wide"
)

# --- 2. ESTILOS PERSONALIZADOS ---
st.markdown("""
    <style>
    .stProgress > div > div > div > div { background-color: #2ecc71; }
    .main { background-color: #f8f9fa; }
    </style>
    """, unsafe_allow_html=True)

# --- 3. CARGA DE MODELO Y DATOS ---
@st.cache_resource
def load_assets():
    # Nombre de tu modelo guardado
    return joblib.load('modelo_futbol_xgboost.pkl')

@st.cache_data
def load_data():
    # Cargamos el dataset y nos aseguramos de que la fecha sea tipo datetime
    df = pd.read_csv('df_final.csv')
    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'])
    return df

modelo_final = load_assets()
df_matches = load_data()

# --- 4. SIDEBAR: SELECCIÓN AUTOMATIZADA ---
with st.sidebar:
    st.title("🛡️ Panel de Control")
    st.subheader("Selección de Partido")
    
    lista_equipos = sorted(df_matches['HomeTeam'].unique())
    
    equipo_local = st.selectbox("Equipo Local", lista_equipos, index=0)
    equipo_visitante = st.selectbox("Equipo Visitante", lista_equipos, index=1)
    
    st.markdown("---")
    
    # APLICANDO TU CORRECCIÓN: Ordenar por fecha antes de sacar el último registro
    stats_h = (
        df_matches[df_matches['HomeTeam'] == equipo_local]
        .sort_values("Date")
        .iloc[-1]
    )

    stats_a = (
        df_matches[df_matches['AwayTeam'] == equipo_visitante]
        .sort_values("Date")
        .iloc[-1]
    )

    # Variables extraídas (coherentes con tus nombres de columna)
    home_elo = stats_h['Homeelo']
    away_elo = stats_a['Awayelo']
    form_home = stats_h['Form5Home']
    form_away = stats_a['Form5Away']
    odd_home = stats_h['OddHome']
    odd_draw = stats_h['OddDraw']
    odd_away = stats_h['OddAway']

    st.success(f"Datos actualizados para el {equipo_local} vs {equipo_visitante}")

# --- 5. LÓGICA DE PREDICCIÓN ---
# El DataFrame de entrada debe tener el mismo orden que cuando entrenaste el modelo
input_df = pd.DataFrame({
    'HomeElo': [home_elo],
    'AwayElo': [away_elo],
    'Form5Home': [form_home],
    'Form5Away': [form_away],
    'OddHome': [odd_home],
    'OddDraw': [odd_draw],
    'OddAway': [odd_away],
    'Anio': [2025], 
    'Mes': [2],
    'Dia_Semana': [5],
    'Hour_sin': [0.5],
    'Hour_cos': [0.8]
})

probs = modelo_final.predict_proba(input_df)[0] 

# --- 6. CUERPO PRINCIPAL (Dashboard Visual) ---
st.title("⚽ AI Football Prediction Dashboard")
st.caption(f"Analizando el enfrentamiento entre {equipo_local} y {equipo_visitante}")

# Métricas destacadas
m1, m2, m3 = st.columns(3)
with m1:
    st.metric(f"Potencia {equipo_local}", int(home_elo), delta=int(home_elo - away_elo))
with m2:
    st.metric("Cuota Empate", odd_draw)
with m3:
    st.metric(f"Potencia {equipo_visitante}", int(away_elo), delta=int(away_elo - home_elo))

st.divider()

col_main1, col_main2 = st.columns([1, 2], gap="large")

with col_main1:
    with st.container(border=True):
        st.subheader("📋 Parámetros detectados")
        resumen = {
            "Variable": ["ELO Local", "ELO Visitante", "Forma (Últ. 5)", "Cuota Local"],
            "Valor": [f"{home_elo:.0f}", f"{away_elo:.0f}", f"{form_home*100:.1f}%", f"{odd_home:.2f}"]
        }
        st.table(pd.DataFrame(resumen))

with col_main2:
    with st.container(border=True):
        st.subheader("🔮 Probabilidades Calculadas")
        
        clases = ['Local (H)', 'Empate (D)', 'Visitante (A)']
        iconos = ['🏠', '🤝', '🚀']
        colores = ["#2ecc71", "#f1c40f", "#e74c3c"]
        
        for i in range(len(clases)):
            col_t, col_p = st.columns([2, 1])
            col_t.write(f"### {iconos[i]} {clases[i]}")
            col_p.write(f"## {probs[i]*100:.1f}%")
            st.progress(probs[i])
        
        st.divider()
        ganador_idx = np.argmax(probs)
        st.success(f"### 🎯 Pronóstico Sugerido: **{clases[ganador_idx]}**")

st.markdown("---")
st.caption("Prototipo de Machine Learning - Análisis Pre-Partido.")